In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42


In [2]:
path = "synthetic_cardiac_dataset_2.csv"   
df = pd.read_csv(path)

df.shape, df.head(5)


((20030, 25),
                    ID  Age Sex      Height  Weight        BMI  Hospital  Department Admission_Date Last_Visit_Date Education_Level          CPT   RestingBP  Cholesterol       MaxHR  CAD  \
 0  2020-11-11_04_03_1   44   1  153.614797    45.0  19.069812  Shariati           3     2020.11.11      2017-06-07     High School  Non-anginal  129.270576   302.432902  188.866223    3   
 1  2020-11-11_01_05_2   63   0  157.441033    45.0  18.154181     Milad           5     2020-11-11             NaN       Secondary      Typical   84.744865   246.525301  140.355652    2   
 2  2020-11-11_02_04_3   63   0  154.433012    45.0  18.868276      Imam           4     2020-11-11             NaN     High School  Non-anginal  157.142272   223.641602  130.787744    1   
 3  2020-11-11_02_02_4   69   1  152.551650    45.0  19.336537      Imam           2     2020.11.11             NaN       Bachelor+  Non-anginal  168.234476   152.410294  198.657664    0   
 4  2020-11-11_03_05_5   47   1  157

In [3]:
print("shape:", df.shape)

# Target
print("\nTarget value counts:")
print(df["Target"].value_counts(dropna=False))

# dtypes
print("\nDtypes:")
print(df.dtypes)

# missing
miss = df.isna().mean().sort_values(ascending=False)
print("\nMissing % (top 30):")
print((miss.head(30)*100).round(2))

# numeric ranges
num_cols = df.select_dtypes(include=[np.number]).columns
print("\nNumeric min/max:")
print(df[num_cols].agg(["min","max"]).T)

shape: (20030, 25)

Target value counts:
Target
1    10020
0    10010
Name: count, dtype: int64

Dtypes:
ID                    object
Age                    int64
Sex                   object
Height               float64
Weight               float64
BMI                  float64
Hospital              object
Department             int64
Admission_Date        object
Last_Visit_Date       object
Education_Level       object
CPT                   object
RestingBP            float64
Cholesterol          float64
MaxHR                float64
CAD                    int64
Heart_Failure         object
Arrhythmia            object
Valvular_Disease      object
CHD                     bool
Diabetes              object
Smoking               object
Alcoholism             int64
Depression_Stress      int64
Target                 int64
dtype: object

Missing % (top 30):
Arrhythmia           63.85
Last_Visit_Date      15.01
Education_Level       5.01
Diabetes              3.00
Smoking               3.00


In [4]:
# Validation rules based on domain logic (initial – will be refined)
validation_rules = {
    "Age": {"min": 0, "max": 120},
    "Height": {"min": 50, "max": 250},      # cm
    "Weight": {"min": 20, "max": 300},      # kg
    "BMI": {"min": 10, "max": 80},
    "RestingBP": {"min": 50, "max": 300},
    "Cholesterol": {"min": 50, "max": 600},
    "MaxHR": {"min": 40, "max": 250},
    "CAD": {"allowed": [0,1,2,3,4]},
    "Target": {"allowed": [0,1]}
}


In [5]:
num_cols = df.select_dtypes(include=[np.number]).columns

df[num_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Age,20030.0,54.329805,9.743665,16.000000,31.000000,38.000000,48.000000,54.000000,61.000000,70.000000,76.000000,85.000000
Height,20030.0,167.963602,7.919339,149.072770,150.000000,154.829218,162.506571,167.852947,173.261808,181.100043,186.647337,200.000000
Weight,20030.0,75.533671,18.827216,45.000000,45.000000,45.000000,61.583682,75.021467,88.356816,107.913429,121.345806,140.000000
BMI,20030.0,26.342019,4.280720,18.077934,18.234507,19.043676,23.315530,26.608445,29.299883,32.752235,37.048805,43.959375
Department,20030.0,2.984923,1.420931,1.000000,1.000000,1.000000,2.000000,3.000000,4.000000,5.000000,5.000000,5.000000
RestingBP,20030.0,129.460557,20.374482,70.000000,82.164997,96.155471,115.500507,129.429135,143.165988,163.016230,176.764179,220.000000
Cholesterol,20030.0,240.805181,45.044152,100.000000,135.694694,166.443403,210.671513,240.781497,271.569132,314.238175,345.167360,407.109530
MaxHR,20030.0,130.837464,40.781076,60.008189,61.423138,67.046707,96.047091,130.651268,165.984804,194.298975,200.781688,209.955804
CAD,20030.0,1.417374,0.992275,0.000000,0.000000,0.000000,1.000000,1.000000,2.000000,3.000000,4.000000,4.000000
Alcoholism,20030.0,25.187768,14.899009,0.000000,0.000000,2.000000,12.000000,25.000000,38.000000,48.000000,50.000000,99.000000


In [6]:
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for c in cat_cols:
    print(f"\n--- {c} ---")
    display(df[c].value_counts(dropna=False))



--- ID ---


ID
2021-04-02_03_02_1476    2
2021-02-24_04_05_1082    2
2021-10-21_02_03_3701    2
2021-06-23_04_01_2384    2
2022-02-23_02_04_5067    2
                        ..
2020-11-13_02_04_28      1
2020-11-13_04_04_29      1
2020-11-13_03_01_30      1
2020-11-13_04_04_31      1
2020-11-13_01_01_32      1
Name: count, Length: 20000, dtype: int64


--- Sex ---


Sex
1    12907
0     5321
M     1259
F      543
Name: count, dtype: int64


--- Hospital ---


...

CHD
False    18214
True      1816
Name: count, dtype: int64


--- Diabetes ---


Diabetes
0        8732
1        8698
NaN       600
True      529
False     508
Yes       505
No        458
Name: count, dtype: int64


--- Smoking ---


Smoking
1        7743
0        7686
False    1021
Yes      1012
No       1001
True      967
NaN       600
Name: count, dtype: int64

In [7]:
def normalize_binary(series):
    s = series.copy()

    # اگر متنی است
    if s.dtype == "O":
        s = s.astype(str).str.strip().str.lower()
        s = s.replace({
            "1": 1, "true": 1, "yes": 1, "y": 1, "بله": 1,
            "0": 0, "false": 0, "no": 0, "n": 0, "خیر": 0
        })

    return s


In [8]:
binary_cols = ["Sex", "Heart_Failure", "Diabetes", "Smoking", "CHD"]

for c in binary_cols:
    if c in df.columns:
        df[c] = normalize_binary(df[c])

for c in binary_cols:
    if c in df.columns:
        print(f"\n{c}")
        display(df[c].value_counts(dropna=False))



Sex


C:\Users\Webhouse\AppData\Local\Temp\ipykernel_7140\3942470430.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = s.replace({


Sex
1    12907
0     5321
m     1259
f      543
Name: count, dtype: int64


Heart_Failure


Heart_Failure
1    10097
0     9933
Name: count, dtype: int64

...

Diabetes
1      9732
0      9698
nan     600
Name: count, dtype: int64


Smoking


Smoking
1      9722
0      9708
nan     600
Name: count, dtype: int64


CHD


CHD
False    18214
True      1816
Name: count, dtype: int64

In [9]:
df["Arrhythmia"] = (
    df["Arrhythmia"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "PVC": "PVC",
        "AF": "AF",
        "SVT": "SVT",
        "NONE": "NONE",
        "NAN": np.nan
    })
)

df["Arrhythmia"].value_counts(dropna=False)


Arrhythmia
NaN     12790
PVC      3381
AF       1748
SVT      1430
NONE      681
Name: count, dtype: int64

In [10]:
df["Valvular_Disease"] = (
    df["Valvular_Disease"]
    .astype(str)
    .str.strip()
    .replace({
        "خفبف": "خفیف",
        "وسط": "متوسط",
        "حاد": "شدید"
    })
)

df["Valvular_Disease"].value_counts(dropna=False)


Valvular_Disease
خفیف     5064
شدید     5059
بدون     4970
متوسط    4937
Name: count, dtype: int64

In [11]:
df["Education_Level"] = df["Education_Level"].fillna("Unknown")

df["Education_Level"].value_counts()


Education_Level
Unknown        4157
High School    3220
Bachelor+      3204
Illiterate     3196
Secondary      3137
Primary        3116
Name: count, dtype: int64

In [12]:
df["Admission_Date"] = pd.to_datetime(df["Admission_Date"], errors="coerce")
df["Last_Visit_Date"] = pd.to_datetime(df["Last_Visit_Date"], errors="coerce")

df[["Admission_Date", "Last_Visit_Date"]].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20030 entries, 0 to 20029
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Admission_Date   5000 non-null   datetime64[ns]
 1   Last_Visit_Date  17024 non-null  datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 313.1 KB


In [13]:
df["Followup_Days"] = (df["Last_Visit_Date"] - df["Admission_Date"]).dt.days

df["Followup_Days"].describe()
df[df["Followup_Days"] < 0].head()


,ID,Age,Sex,Height,Weight,BMI,Hospital,Department,Admission_Date,Last_Visit_Date,Education_Level,CPT,RestingBP,Cholesterol,MaxHR,CAD,Heart_Failure,Arrhythmia,Valvular_Disease,CHD,Diabetes,Smoking,Alcoholism,Depression_Stress,Target,Followup_Days
0,2020-11-11_04_03_1,44,1,153.614797,45.000000,19.069812,Shariati,3,2020-11-11,2017-06-07,High School,Non-anginal,129.270576,302.432902,188.866223,3,0,AF,متوسط,False,0,1,34,8,1,-1253.0
7,2020-11-11_04_05_8,47,1,157.697152,45.000000,18.095260,Shariati,5,2020-11-11,2019-04-18,Unknown,Atypical,140.295203,303.403695,128.185226,3,0,SVT,بدون,False,0,0,28,6,1,-573.0
8,2020-11-11_03_01_9,56,1,154.173593,45.000000,18.931827,Modarres,1,2020-11-11,2019-06-28,Bachelor+,Non-anginal,153.506834,272.249098,104.568334,2,1,NaN,متوسط,False,1,0,48,7,1,-502.0
11,2020-11-12_02_01_12,30,0,154.555154,45.996215,19.255514,Imam,1,2020-11-12,2018-06-03,High School,Typical,115.922532,236.907293,118.763651,1,0,NaN,بدون,False,0,0,50,1,0,-893.0
13,2020-11-12_02_01_14,65,0,152.551252,45.000000,19.336638,Imam,1,2020-11-12,2017-07-02,Secondary,Typical,146.817632,251.143283,115.963200,2,1,NaN,خفیف,False,1,1,23,1,1,-1229.0


In [14]:
issue_log = []

# Age خارج از بازه
age_out = (df["Age"] > 80) | (df["Age"] < 15)
issue_log.append({
    "feature": "Age",
    "issue": "out_of_range",
    "count": int(age_out.sum())
})

# MaxHR خارج از بازه
maxhr_out = df["MaxHR"] > 200
issue_log.append({
    "feature": "MaxHR",
    "issue": "out_of_range",
    "count": int(maxhr_out.sum())
})

# Alcoholism مقدار 99
alc_out = df["Alcoholism"] > 50
issue_log.append({
    "feature": "Alcoholism",
    "issue": "invalid_value_>50",
    "count": int(alc_out.sum())
})

pd.DataFrame(issue_log)


,feature,issue,count
0,Age,out_of_range,1
1,MaxHR,out_of_range,201
2,Alcoholism,invalid_value_>50,40


In [15]:
for c in ["Sex", "Heart_Failure", "Diabetes", "Smoking", "CHD"]:
    print(c)
    display(df[c].value_counts(dropna=False))


Sex


Sex
1    12907
0     5321
m     1259
f      543
Name: count, dtype: int64

Heart_Failure


Heart_Failure
1    10097
0     9933
Name: count, dtype: int64

Diabetes


Diabetes
1      9732
0      9698
nan     600
Name: count, dtype: int64

Smoking


Smoking
1      9722
0      9708
nan     600
Name: count, dtype: int64

CHD


CHD
False    18214
True      1816
Name: count, dtype: int64

In [16]:
# اصلاح Sex
df["Sex"] = (
    df["Sex"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"m": 1, "f": 0})
)

# اصلاح CHD
df["CHD"] = df["CHD"].replace({True: 1, False: 0})


C:\Users\Webhouse\AppData\Local\Temp\ipykernel_7140\1740326434.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["CHD"] = df["CHD"].replace({True: 1, False: 0})


In [17]:
for c in ["Sex", "CHD"]:
    print(c)
    display(df[c].value_counts(dropna=False))


Sex


Sex
1    12907
0     5321
1     1259
0      543
Name: count, dtype: int64

CHD


CHD
0    18214
1     1816
Name: count, dtype: int64

In [18]:
df["Arrhythmia"].value_counts(dropna=False)


Arrhythmia
NaN     12790
PVC      3381
AF       1748
SVT      1430
NONE      681
Name: count, dtype: int64

In [19]:
df["Valvular_Disease"].value_counts(dropna=False)


Valvular_Disease
خفیف     5064
شدید     5059
بدون     4970
متوسط    4937
Name: count, dtype: int64

In [20]:
df["Education_Level"].value_counts(dropna=False)


Education_Level
Unknown        4157
High School    3220
Bachelor+      3204
Illiterate     3196
Secondary      3137
Primary        3116
Name: count, dtype: int64

In [21]:
df[["Admission_Date", "Last_Visit_Date"]].isna().sum()



Admission_Date     15030
Last_Visit_Date     3006
dtype: int64

In [22]:
df["Followup_Days"].describe()


count    4244.000000
mean    -1002.417766
std       471.494204
min     -2337.000000
25%     -1404.000000
50%     -1010.000000
75%      -590.000000
max      1169.000000
Name: Followup_Days, dtype: float64

In [23]:
df[df["Followup_Days"] < 0].shape


(4243, 26)

In [24]:
pd.DataFrame(issue_log)


,feature,issue,count
0,Age,out_of_range,1
1,MaxHR,out_of_range,201
2,Alcoholism,invalid_value_>50,40


In [25]:
# Age
df.loc[df["Age"] > 80, "Age"] = 80

# MaxHR
df.loc[df["MaxHR"] > 200, "MaxHR"] = 200

# Alcoholism
df.loc[df["Alcoholism"] > 50, "Alcoholism"] = 0


In [26]:
df["Arrhythmia"] = df["Arrhythmia"].fillna("NONE")
df["Arrhythmia"].value_counts()


Arrhythmia
NONE    13471
PVC      3381
AF       1748
SVT      1430
Name: count, dtype: int64

In [27]:
df["Diabetes"] = df["Diabetes"].fillna(0)
df["Diabetes"].value_counts()


Diabetes
1      9732
0      9698
nan     600
Name: count, dtype: int64

In [28]:
df["Smoking"] = df["Smoking"].fillna(0)
df["Smoking"].value_counts()


Smoking
1      9722
0      9708
nan     600
Name: count, dtype: int64

In [29]:
df.isna().sum().sort_values(ascending=False)


Followup_Days        15786
Admission_Date       15030
Last_Visit_Date       3006
Age                      0
Sex                      0
ID                       0
BMI                      0
Weight                   0
Department               0
Height                   0
Hospital                 0
Education_Level          0
RestingBP                0
CPT                      0
MaxHR                    0
CAD                      0
Heart_Failure            0
Cholesterol              0
Arrhythmia               0
Valvular_Disease         0
Diabetes                 0
CHD                      0
Smoking                  0
Alcoholism               0
Depression_Stress        0
Target                   0
dtype: int64

In [30]:

df["Last_Visit_Date"] = df["Last_Visit_Date"].fillna(df["Admission_Date"])


df = df.drop(columns=["Admission_Date", "Followup_Days"])


In [31]:
df.isna().sum()


ID                      0
Age                     0
Sex                     0
Height                  0
Weight                  0
BMI                     0
Hospital                0
Department              0
Last_Visit_Date      2250
Education_Level         0
CPT                     0
RestingBP               0
Cholesterol             0
MaxHR                   0
CAD                     0
Heart_Failure           0
Arrhythmia              0
Valvular_Disease        0
CHD                     0
Diabetes                0
Smoking                 0
Alcoholism              0
Depression_Stress       0
Target                  0
dtype: int64

In [32]:
df = df.drop(columns=["Last_Visit_Date"])
df.isna().sum()


ID                   0
Age                  0
Sex                  0
Height               0
Weight               0
BMI                  0
Hospital             0
Department           0
Education_Level      0
CPT                  0
RestingBP            0
Cholesterol          0
MaxHR                0
CAD                  0
Heart_Failure        0
Arrhythmia           0
Valvular_Disease     0
CHD                  0
Diabetes             0
Smoking              0
Alcoholism           0
Depression_Stress    0
Target               0
dtype: int64

In [33]:
TARGET_COL = "Target"

y = df[TARGET_COL].copy()
X = df.drop(columns=["ID", TARGET_COL])


In [34]:
education_map = {
    "Illiterate": 0,
    "Primary": 1,
    "Secondary": 2,
    "High School": 3,
    "Bachelor+": 4,
    "Unknown": 2  
}

valvular_map = {
    "بدون": 0,
    "خفیف": 1,
    "متوسط": 2,
    "شدید": 3
}


In [35]:
X["Education_Level"] = X["Education_Level"].map(education_map)
X["Valvular_Disease"] = X["Valvular_Disease"].map(valvular_map)

X[["Education_Level", "Valvular_Disease"]].describe()


,Education_Level,Valvular_Disease
count,20030.000000,20030.000000
mean,2.005991,1.503495
std,1.262716,1.118681
min,0.000000,0.000000
25%,1.000000,1.000000
50%,2.000000,1.000000
75%,3.000000,3.000000
max,4.000000,3.000000


In [36]:
nominal_cols = ["Hospital", "CPT", "Arrhythmia", "Department"]

X = pd.get_dummies(X, columns=nominal_cols, drop_first=False)


In [37]:
X.shape, y.shape


((20030, 34), (20030,))

In [38]:
X.dtypes.value_counts()


bool       17
int64       8
float64     6
object      3
Name: count, dtype: int64

In [39]:
X.select_dtypes(include=["object"]).columns


Index(['Sex', 'Diabetes', 'Smoking'], dtype='object')

In [40]:
for c in ["Sex", "Diabetes", "Smoking"]:
    X[c] = pd.to_numeric(X[c], errors="coerce").astype("Int64")  

X[["Sex", "Diabetes", "Smoking"]] = X[["Sex", "Diabetes", "Smoking"]].fillna(0).astype(int)


In [41]:
X.dtypes.value_counts(), X[["Sex","Diabetes","Smoking"]].head()


(bool       17
 int64      11
 float64     6
 Name: count, dtype: int64,
    Sex  Diabetes  Smoking
 0    1         0        1
 1    0         0        0
 2    0         0        1
 3    1         1        0
 4    1         1        0)

In [42]:
bool_cols = X.select_dtypes(include=["bool"]).columns
X[bool_cols] = X[bool_cols].astype(int)

X.dtypes.value_counts()


int64      28
float64     6
Name: count, dtype: int64

In [43]:
final_df = X.copy()
final_df["Target"] = y.values
final_df.to_csv("final_clean.csv", index=False)
final_df.shape


(20030, 35)